# Chapter 27 — Adaptive Defense: Unknown 공격 자동 학습·자동 대응 (실습)

> **Chapter 27 · Part IV**

ch24 lab 위에 RAG corpus 자동 갱신 + Active Learning Loop + 7 환각 격리 가드를 더한다.

## 주의
- 기본 실행은 합성 데이터 시뮬레이션이며 Docker가 필요하지 않습니다. 실제 표적 연동은 별도 격리 환경과 승인이 필요합니다.
- 실 운영 적용은 §7 의 7개 가드를 모두 통과한 후에만 가능.
- Anthropic API가 없으면 `USE_MOCK_LLM = True`로 mock 응답을 사용합니다. Ollama 연동과는 별개입니다.

## 실습 흐름 (9 Step)
1. 합성 실습 환경 설정 (컨테이너 통합은 확장 설계)
2. RAG corpus 초기 상태 점검
3. NVD daily cron 시뮬레이션
4. 결정론적 합성 변형 50건 비교
5. SecurityAssistant 분석 — corpus 갱신 전/후 비교
6. 미탐 사람 라벨 → ChromaDB patch
7. SGDClassifier `partial_fit` 으로 재학습
8. 7-day 사이클 시뮬레이션 — 탐지율 시계열
9. 7 가드 stress test

> **이 노트북의 7-day 곡선은 결정론적 교육용 illustration 입니다.** 탐지 판정은 LLM 이 아니라
> **Day-1 keyword 시그니처 + corpus 에 patch 된 시그니처의 규칙 매칭**으로 계산하므로, API 키가
> 있어도 실행마다 같은 곡선이 나옵니다. 곡선은 절대 탐지 성능이 아니라 **메커니즘**(corpus 성장 →
> 미탐 재학습 → 탐지 개선, RAG/patch 미사용 대조군은 평탄)을 보이는 것이 목적입니다.
>
> LLM 의 의미 추론(`analyze_unknown`)은 규칙·corpus 가 못 밝힌 변형을 다루는 **별도 데모**로
> 남겨 두었고(Step 4 끝), 이 부분만 실 API 로 동작합니다. corpus 공유 self-heal 의 정량 효과는
> 저장소 `labs/run_lab.py`(RAG 사용 40%→100% vs 미사용 40% 평탄) 가 독립적으로 재현합니다.


## Step 0. 환경 설정 + 패키지 설치

In [ ]:
# === 한글 폰트 설정 (matplotlib) ===
from pathlib import Path
import matplotlib
from matplotlib import font_manager
for fp in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
           '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf']:
    if Path(fp).exists():
        font_manager.fontManager.addfont(fp)
        matplotlib.rcParams['font.family'] = font_manager.FontProperties(fname=fp).get_name()
        break
matplotlib.rcParams['axes.unicode_minus'] = False

In [ ]:
# !pip install chromadb sentence-transformers scikit-learn jsonschema anthropic ollama requests matplotlib pandas
# !pip install --quiet --upgrade pip

In [ ]:
import os, json, time, random, hashlib, pickle
from pathlib import Path
from datetime import date, timedelta, datetime
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# === 환경 변수 ===
DATA_DIR = Path('/tmp/ch27_lab')
DATA_DIR.mkdir(exist_ok=True)
CHROMA_PATH = DATA_DIR / 'chroma'
MODEL_PATH = DATA_DIR / 'rf_model.pkl'
AUDIT_LOG = DATA_DIR / 'audit.jsonl'
ROLLBACK_LOG = DATA_DIR / 'rollback.jsonl'

# Mock 모드 — Anthropic API key 없으면 True
USE_MOCK_LLM = not bool(os.environ.get('ANTHROPIC_API_KEY'))
print(f'USE_MOCK_LLM = {USE_MOCK_LLM}')
print(f'DATA_DIR = {DATA_DIR}')

---
## Step 1. RAG corpus 초기 상태

ch24 lab 시작 시점의 corpus 를 가정한다 — known CVE 100건 + MITRE ATT&CK 50건.

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: 기존 collection 정리 후 새로 시작

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
import chromadb
from chromadb.config import Settings

client = ...
# 기존 collection 정리 후 새로 시작
try:
    ...
except Exception:
    pass
col = ...

# 초기 corpus — known CVE 5건만 시드 (mock)
SEED_CORPUS = ...
    ...
for cid, desc, meta in SEED_CORPUS:
    col.add(documents = ...

...
print('source 분포:', Counter(m.get('source') for m in col.get(include = ...
```
</details>


---
## Step 2. NVD daily cron 시뮬레이션

본 실습은 NVD API 호출 대신 **mock 신규 CVE 5건** 을 인덱싱한다 (오프라인 환경 지원).
실제 운영에서는 `requests.get('https://services.nvd.nist.gov/rest/json/cves/2.0')` 호출.

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: _nvd_seq = ...


<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
_nvd_seq = ...
def update_nvd_mock(target_date: ...
    ...
    descs = ...
        ...
    added = ...
    for _ in range(count):
        _nvd_seq += ...
        cid = ...
        ...
            documents = ...
            ids = ...
            metadatas = ...
                        ...
        added += ...
    return ...

...
added = ...
...
```
</details>


---
## Step 3. 공격자 LLM 의 의미 변형 payload 50건 생성

실 운영에서는 공격자 LLM(Ollama 등) 이 매 시도마다 페이로드를 동적으로 변형하지만, 이 노트북은
**곡선을 실행마다 재현**하기 위해 변형 50건을 결정론 데이터셋으로 고정한다. 각 payload 는
(문자열, 실제 공격 계열) 로 태깅하며, 5 개 원본에서 파생한 plain/난독화/의미우회 변형을 섞는다.


In [ ]:
BASE_PAYLOADS = [
    "' OR 1=1--",
    '<script>alert(1)</script>',
    '${jndi:ldap://x.x/y}',
    'cmd.exe /c whoami',
    '../../../etc/passwd',
]

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: 공격자 LLM 이 만드는 변형을 결정론 데이터셋으로 고정한다(재현성). 세 종류를 섞는다:


<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
# 공격자 LLM 이 만드는 변형을 결정론 데이터셋으로 고정한다(재현성). 세 종류를 섞는다:
#  - plain: Day-1 시그니처 엔진이 아는 교과서형 (원본 그대로 잡힘)
#  - 난독화: 인라인 주석·URL 인코딩·문자 치환 (Day-1 은 놓치지만, 정규화하면 계열이 드러나 학습 가능)
#  - 의미 우회: 이중 인코딩·HTML 엔티티 (규칙 정규화로도 계열 미확정 → 사람/LLM 판단 영역)
MUTATIONS = ...
    # --- plain (30) : Day-1 keyword 시그니처가 잡는 교과서형 ---
    ("' OR 1 = ...
    ("' OR 1 = ...
    ...
    ("admin' OR '1' = ...
    ("' OR 1 = ...
    ...
    ("file = ...
    ...
    ("path = ...
    ...
    # --- 난독화 (18) : Day-1 은 미탐, 정규화하면 계열이 드러나 사람 라벨→학습 가능 ---
    ...
    # --- 의미 우회 (2) : 규칙 정규화로도 계열 미확정 → LLM 의미 추론 영역 ---
    ...
mutations = ...
...
for p, fam in MUTATIONS[: ...
    ...
```
</details>


---
## Step 4. Day-1 탐지 기준선 — keyword 단독 vs corpus 결합

탐지 판정을 **결정론**으로 정의한다: `탐지 = Day-1 keyword 시그니처 매치 OR corpus 에 patch 된
시그니처 매치`. Day-1 시그니처 엔진은 교과서형만 알아 난독화 변형을 놓친다. 아직 Active Learning
전이라 corpus 에 patch 된 시그니처가 없으므로 두 경로의 탐지율은 같고, 여기서 나온 미탐이 이후
Step 5~7 의 학습 재료가 된다. LLM 의 의미 추론은 이 셀 뒤 별도 데모로 분리했다.


In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: ATT&CK whitelist — LLM 이 만든 가짜 T-ID 를 걸러내는 데 쓴다(Step 9).


<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
import re, urllib.parse

# ATT&CK whitelist — LLM 이 만든 가짜 T-ID 를 걸러내는 데 쓴다(Step 9).
ALLOWED_ATTACK_IDS = ...
                      ...
FAMILY_TO_ATTACK = ...
                    ...

# ── Day-1 시그니처 엔진(결정론) ───────────────────────────────────────
# 처음부터 아는 "교과서형" 토큰만 매칭한다. 난독화 변형은 일부러 놓치게 두어,
# 매일 사람 라벨→corpus patch 로 학습해 탐지율이 오르는 과정을 관측한다.
DAY1_SIGNATURES = ...
    'SQLi': ["' or 1 = ...
    ...

def _normalize(payload: ...
    ...
    s = ...
    try:
        s = ...
    except Exception:
        pass
    s = ...
    s = ...
    s = ...
    s = ...
    s = ...
    s = ...
    return ...

def keyword_detect(payload: ...
    ...
    p = ...
    for fam, sigs in DAY1_SIGNATURES.items():
        if any(sig in p for sig in sigs):
            return ...
    return ...

def auto_label(payload: ...
    ...
    ns = ...
    fam = ...
    return ...

# 사람 라벨로 학습한 정규화 시그니처 → 계열 (Active Learning corpus patch 의 결정론 표현)
LEARNED = ...

def detect(payload: ...
    """탐지 = ...
    fam = ...
    if fam:
        return ...
    ns = ...
    if ns in LEARNED:
        return ...
    return ...

def detect_rate(payloads, use_corpus: ...
    if use_corpus:
        return ...
    return ...

# ── LLM 의미 추론(별도 데모용, 결정론 곡선과 무관) ────────────────────
def _parse_llm_json(raw: ...
    ...
    text = ...
    m = ...
    if m:
        text = ...
    return ...

def analyze_unknown(payload: ...
    ...
    hits_ids, hits_docs = ...
    if use_rag:
        hits = ...
        hits_docs = ...
        hits_ids = ...
                    for m, i in zip(hits['metadatas'][0], hits['ids'][0])]
    if USE_MOCK_LLM:
        # mock — 결정론 detect() 결과에 corpus citation 을 붙여 흉내낸다.
        fam = ...
        if not fam:
            return ...
                    ...
        return ...
                ...
    # 실제 Anthropic API 경로 (claude-opus-5, 짧은 JSON 판정이라 thinking 비활성)
    import anthropic
    client_ai = ...
    ctx = ...
    resp = ...
        model = ...
        thinking = ...
        system = ...
            ...
        messages = ...
    ...
    try:
        # 첫 블록이 thinking 일 수 있으므로 text 블록만 모아 parse 한다.
        parsed = ...
        if not isinstance(parsed, dict) or parsed.get('category') not in {'UNKNOWN','BENIGN','SQLi','XSS','RCE','CMDi','PathTraversal'}:
            raise ...
        if type(parsed.get('severity')) is not int or not 1 <= parsed['severity'] <= 5:
            raise ...
        if not isinstance(parsed.get('citation'), list) or any(cid not in hits_ids for cid in parsed['citation']):
            raise ...
        sev = ...
        sev = ...
        citation = ...
        return ...
            ...
    except Exception:
        return ...
                ...
```
</details>


In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: rate_keyword = ...


<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
rate_keyword = ...
rate_corpus = ...

...

missed0 = ...
...
for p in missed0[: ...
    ...
```
</details>


In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: LLM 의미 추론 데모 (결정론 곡선과 분리). Day-1 규칙·corpus 가 놓친 변형을 LLM 이 어떻게


<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
# LLM 의미 추론 데모 (결정론 곡선과 분리). Day-1 규칙·corpus 가 놓친 변형을 LLM 이 어떻게
# 판정하는지 보인다. USE_MOCK_LLM=False 면 실제 claude-opus-5 를 호출하므로 문구는 실행마다 다를 수 있다.
demo_targets = ...
print(f'USE_MOCK_LLM = ...
for p in demo_targets:
    try:
        v = ...
        ...
        print(f'    category = ...
        print(f'    reasoning = ...
    except Exception as e:
        ...
```
</details>


---
## Step 5. 미탐 사람 라벨 → ChromaDB patch

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: def find_missed(payloads):


<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
def find_missed(payloads):
    ...
    return ...

_patch_seq = ...
def patch_corpus(payload: ...
    ...
    al = ...
    if not al:
        return ...
    ns, fam = ...
    LEARNED[ns] = ...
    _patch_seq += ...
    rid = ...
    col.upsert(documents = ...
               metadatas = ...
                           ...
    return ...

missed = ...
...
labeled = ...
for p in missed:
    fam = ...
    if fam:
        labeled += ...
        if labeled <= 3:
            ...
    else:
        escalated += ...
...
```
</details>


---
## Step 6. SGDClassifier `partial_fit` 으로 ML 1차 재학습

scikit-learn 의 SGDClassifier 는 `partial_fit` 을 직접 지원. RandomForest의 warm_start는 기존 트리의 증분 재학습과 다릅니다.

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: 초기 학습 데이터 (소량 — Day 1)


<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
from sklearn.linear_model import SGDClassifier
from sklearn.feature_extraction.text import HashingVectorizer

# 초기 학습 데이터 (소량 — Day 1)
X_init = ...
          "' OR 1 = ...
y_init = ...
vec = ...
model = ...
model.partial_fit(vec.transform(X_init), y_init, classes = ...
...

# 미탐 케이스 추가 학습 (missed 는 문자열 리스트)
missed_payloads = ...
missed_labels = ...
if missed_payloads:
    # HashingVectorizer.transform([]) 은 빈 iterable 에서 StopIteration 을 던진다 —
    # 미탐이 없는 라운드에는 재학습할 데이터가 없으므로 건너뛴다.
    ...
else:
    ...

with open(MODEL_PATH, 'wb') as f:
    ...
```
</details>


---
## Step 7. 7-day 사이클 시뮬레이션 — 탐지율 시계열

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: 1) 오늘의 변형 분석 — 탐지 = keyword OR corpus(patch) 시그니처 (결정론, 패치 전 상태로 측정)


<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
...
PATCH_PER_DAY = ...

history = ...
for day in range(1, 8):
    ...

    # 1) 오늘의 변형 분석 — 탐지 = keyword OR corpus(patch) 시그니처 (결정론, 패치 전 상태로 측정)
    detected = ...
    rate = ...
    # 2) 대조군: corpus/patch 미사용(RAG off) — Day-1 keyword 시그니처만
    rate_off = ...

    # 3) 미탐 → 사람 라벨 → corpus patch (하루 최대 PATCH_PER_DAY 건)
    missed_today = ...
    patched = ...
    for p in missed_today:
        if patched >= PATCH_PER_DAY:
            break
        if patch_corpus(p):
            patched += ...

    ...
    print(f'Day {day}: detected = ...
          f'RAG off = ...
          f'corpus = ...
```
</details>


In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: days = ...


<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
days = ...
rates = ...
rates_off = ...
corpus = ...

fig, (ax1, ax2) = ...
ax1.plot(days, rates, marker = ...
ax1.plot(days, rates_off, marker = ...
         label = ...
...
ax1.legend(); ax1.grid(alpha = ...

ax2.bar(days, corpus, color = ...
...
ax2.grid(alpha = ...

...
```
</details>


---
## Step 8. 7 가드 stress test

환각 격리 가드들이 적대적 입력에 대응하는지 검증.

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: Stress test — (이름, payload, 주입할 가짜 attack_id, 기대 category)


<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
def gate_hitl(verdict: ...
    ...
    if verdict.get("requires_review") or verdict.get("category") == "UNKNOWN":
        return ...
    if verdict.get("severity", 1) >= severity_thresh:
        if verdict.get("confidence", 0.0) < 0.8 or not verdict.get("citation"):
            return ...
        return ...
    return ...

def gate_attack_whitelist(verdict: ...
    ...
    aid = ...
    if aid and aid not in ALLOWED_ATTACK_IDS:
        verdict['attack_id'] = ...
        verdict['reasoning'] = ...
    return ...

def gate_citation_enforce(verdict: ...
    ...
    result = ...
    if result.get("category") == "UNKNOWN" or (
            ...
        result["requires_review"] = ...
    return ...

def build_verdict(payload: ...
    ...
    fam = ...
    if not fam:
        return ...
                ...
    hits = ...
    cites = ...
             for m, i in zip(hits['metadatas'][0], hits['ids'][0])]
    return ...
            ...

# Stress test — (이름, payload, 주입할 가짜 attack_id, 기대 category)
STRESS_CASES = ...
    ('정상 검색',          'q = ...
    ...
    ('가짜 CVE 인용 유도', "' OR 1 = ...
    ...

results_stress = ...
for name, payload, inject_aid, expected in STRESS_CASES:
    v = ...
    if inject_aid: ...
        v['attack_id'] = ...
    v = ...
    v = ...
    gate = ...
    print(f'  [{name}] cat = ...
          f'aid = ...
    ...
```
</details>


---
## Step 9. safe_block + TTL 5분 + rollback log

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: stress test 결과 중 auto_block 인 케이스 차단 시뮬레이션

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
def safe_block(ip: ...
    ...
    entry = ...
        ...
    with ROLLBACK_LOG.open('a') as f:
        ...
    if dry_run:
        ...
    return ...

# stress test 결과 중 auto_block 인 케이스 차단 시뮬레이션
for r in results_stress:
    if r['gate'] == 'auto_block_with_ttl':
        safe_block(ip = ...
                   ttl_sec = ...

# rollback log 확인
...
if ROLLBACK_LOG.exists():
    for line in ROLLBACK_LOG.read_text().splitlines()[-5: ...
        e = ...
        ...
              f'ip = ...
```
</details>


---
## 챌린지

1. **CISA KEV feed 추가**: NVD 외에 `cisa.gov/known-exploited-vulnerabilities-catalog` 를 일별 인덱싱하고, KEV 등재 CVE 에 가중치 +0.1 부여.
2. **drift sentinel 5분 윈도우**: 위 KS(Kolmogorov-Smirnov) 검정 윈도우를 5분으로 줄이고, LLM 호출 빈도가 증가했을 때 비용·정확도 trade-off 측정.
3. **Catastrophic forgetting 측정**: `partial_fit` 으로 누적 학습한 모델이 Day 1 기존 known CVE 정탐율을 유지하는지 측정.
4. **공격자 LLM 강화**: Ollama 가 설치되어 있다면 `MUTATIONS` 데이터셋을 Llama 3.2 3B 의 동적 생성으로 교체하고, 변형 다양성·탐지율 변화를 비교. (동적 생성으로 바꾸면 곡선은 더 이상 결정론이 아니다.)
5. **의미 우회 2건 자동화**: 규칙 정규화가 놓친 이중 인코딩·HTML 엔티티 변형을 `analyze_unknown` 의 LLM 의미 추론으로 잡고, corpus 에 patch 해 탐지율을 96%→100% 로 올려 본다.


---
## 정리
- **4 소스 daily cron** 으로 신규 CVE 가 하루 안에 corpus 에 반영되는 구조를 만들었습니다.
- **citation enforcement + ATT&CK whitelist** 로 LLM 환각이 곧바로 운영 차단으로 이어지지 않게 막았습니다.
- **Active Learning** 으로 라벨링한 미탐 패턴을 다음 round 부터 탐지하도록 재학습했습니다.
- **7-day 사이클**에서 탐지율이 60%(Day 1)에서 96%(Day 7)로 매일 오르는 과정을 결정론으로 재현했습니다 — corpus/patch 를 끈 대조군은 60% 로 평탄합니다(교육용 illustration, 절대 성능이 아님).
- zero-day·환각 우회·사회공학은 여전히 사람이 판단해야 하는 영역입니다(의미 우회 2건이 그 예입니다).
- 구성 요소는 Ollama·Llama·nmap·Zeek·ChromaDB·sentence-transformers·scikit-learn·iptables·Docker 로 모두 무료이고, 유료인 Anthropic API 도 Ollama Llama 로 대체할 수 있습니다(판정 품질은 떨어집니다).

> **다음 챕터:** 운영 거버넌스 — 격리 lab 의 자동 방어를 실제 SOC 에 단계적으로 도입하는 절차.
